In [1]:
import pandas as pd
import re
import json
import tqdm

### READ META INFO

In [2]:
# gene info
# columns: uni_id, name, cg_id (used in GEM/PPI), Cgl_id (used in KEGG KO), full_name
gene_info = pd.read_csv('../../../raw/CGL/cgl_gene_info.txt', sep='\t',
                        names=['uni_id', 'name', 'cg_id', 'Cgl_id', 'full_name'])
name2uniid = {}   # NOTE: use lower case
id2uniid = {}     # cg_id -> uniid  (for GEM gene rules and PPI)
cgl_id2uniid = {} # Cgl_id -> uniid (for KEGG KO triples)
all_gene_uniid = set(gene_info['uni_id'].unique())
for i in range(gene_info.shape[0]):
    uniid  = gene_info.iloc[i]['uni_id']
    name   = gene_info.iloc[i]['name']
    cg_id  = gene_info.iloc[i]['cg_id']
    Cgl_id = gene_info.iloc[i]['Cgl_id']
    if pd.notna(name):
        name2uniid[name.lower()] = uniid
    if pd.notna(cg_id):
        id2uniid[cg_id] = uniid
    if pd.notna(Cgl_id):
        cgl_id2uniid[Cgl_id] = uniid

In [3]:
# reaction info
rxn_info_df = pd.read_csv('../../../raw/shared/reaction_info.txt', sep='\t')
rxnbigg = {}
for i in range(rxn_info_df.shape[0]):
    new_bigg = rxn_info_df.iloc[i]['bigg_id']
    old_bigg = rxn_info_df.iloc[i]['old_bigg_ids'].split('; ')
    old_bigg.append(new_bigg)
    for x in old_bigg:
        rxnbigg[x] = new_bigg

In [4]:
# metabolite info
met_info_df = pd.read_csv('../../../raw/shared/metabolite_info.txt', sep='\t')
metabigg = {}
for i in range(met_info_df.shape[0]):
    new_bigg = met_info_df.iloc[i]['bigg_id']
    try:
        old_bigg = met_info_df.iloc[i]['old_bigg_ids'].split('; ')
        old_bigg.append(new_bigg)
    except:
        old_bigg = [new_bigg]
    for x in old_bigg:
        metabigg[x] = new_bigg

inchikey2id = {}
for i in range(met_info_df.shape[0]):
    met_bigg_id = met_info_df.iloc[i]['bigg_id']
    if met_bigg_id.endswith('_c'):
        inchikey = met_info_df.iloc[i]['inchikey']
        if pd.isna(inchikey):
            continue
        inchikey2id['-'.join(inchikey.split('-')[:-1])] = met_bigg_id

In [5]:
# 流通代谢物
common_meta_df = pd.read_csv('../../../raw/shared/common_metabolite.txt', sep='\t')
common_meta = list(common_meta_df['met_id_in_1515'].unique())
common_meta = [metabigg[x] if x in metabigg.keys() else x for x in common_meta]
# strip compartment suffix to match metabolites in any compartment (_m, _x, _e, etc.)
common_meta_base = {m.rsplit('_', 1)[0] for m in common_meta}

In [6]:
# GEM model (iCW773)
with open('../../../raw/CGL/iCW773.json', 'r', encoding='utf-8') as file:
    gem = json.load(file)

# Merge Cgl_id -> uni_id into id2uniid so that GEM gene rules using Cgl-format IDs are resolved
# (iCW773 gene_reaction_rule may contain both cg_id and Cgl_id format locus tags)
for cgl_id, uni in cgl_id2uniid.items():
    if cgl_id not in id2uniid:
        id2uniid[cgl_id] = uni

# Supplement id2uniid via string_name.txt preferred_name -> name2uniid
# This covers cg_ids absent from cgl_gene_info.txt but whose preferred gene name is known
str_name_df = pd.read_csv('../../../raw/CGL/string_name.txt', sep='\t')
str_name_df.columns = ['string_id', 'preferred_name', 'protein_size', 'annotation']
for _, row in str_name_df.iterrows():
    cg_id = row['string_id'].split('.')[1]
    if cg_id not in id2uniid:
        pref = str(row['preferred_name']).lower()
        if pref in name2uniid:
            id2uniid[cg_id] = name2uniid[pref]

# Also supplement with UniProt annotations embedded in the GEM itself
for g in gem['genes']:
    if g['id'] not in id2uniid and 'annotation' in g and 'uniprot' in g['annotation']:
        prot_uni = g['annotation']['uniprot']
        id2uniid[g['id']] = prot_uni
        all_gene_uniid.add(prot_uni)

### GENERATE TRIPLES

In [7]:
# reaction triples: Reaction --Produces/Consumes--> Metabolite (skip common metabolites, skip BIOMASS)
# reversible reactions (lower_bound < 0) are expanded with a _-1 reverse direction
triples = []

for x in tqdm.tqdm(gem['reactions']):
    if 'BIOMASS' in x['id']:
        continue
    rxn_id = rxnbigg.get(x['id'], x['id'])
    reversible = x.get('lower_bound', 0) < 0

    for k, v in x['metabolites'].items():
        k = metabigg.get(k, k)
        if k in common_meta or k.rsplit('_', 1)[0] in common_meta_base:
            continue
        if v > 0:
            triples.append(('Reaction:' + rxn_id, 'Produces', 'Metabolite:' + k))
            if reversible:
                triples.append(('Reaction:' + rxn_id + '_-1', 'Consumes', 'Metabolite:' + k))
        elif v < 0:
            triples.append(('Reaction:' + rxn_id, 'Consumes', 'Metabolite:' + k))
            if reversible:
                triples.append(('Reaction:' + rxn_id + '_-1', 'Produces', 'Metabolite:' + k))

100%|██████████| 1194/1194 [00:00<00:00, 100276.30it/s]


In [8]:
# catalyze triples: Protein --Catalyzes--> Reaction
# 'and' rules are expanded to individual proteins
# reversible reactions also get a _-1 entry
# skip BIOMASS and reactions where all metabolites are common (no specific metabolites)
# NOTE: CGL gene IDs in iCW773 may be cg_id format (e.g. cg0681) or Cgl_id format (e.g. Cgl2109)
for x in tqdm.tqdm(gem['reactions']):
    if 'BIOMASS' in x['id']:
        continue
    rxn_id = rxnbigg.get(x['id'], x['id'])
    reversible = x.get('lower_bound', 0) < 0

    if x['gene_reaction_rule'] == '':
        continue
    mets_mapped = [metabigg.get(k, k) for k in x['metabolites']]
    if all(m in common_meta or m.rsplit('_', 1)[0] in common_meta_base for m in mets_mapped):
        continue

    for rule_token in x['gene_reaction_rule'].split(' or '):
        if 'and' in rule_token:
            locus_tags = re.findall(r'(?:cg|Cgl)\d+', rule_token)
            proteins = [id2uniid[lt] for lt in locus_tags if lt in id2uniid]
        else:
            locus_tag = rule_token.strip().strip('()')
            try:
                proteins = [id2uniid[locus_tag]]
            except:
                continue
        for prot in proteins:
            triples.append(('CGL_Protein:' + prot, 'Catalyzes', 'Reaction:' + rxn_id))
            if reversible:
                triples.append(('CGL_Protein:' + prot, 'Catalyzes', 'Reaction:' + rxn_id + '_-1'))

100%|██████████| 1194/1194 [00:00<00:00, 195159.93it/s]


In [9]:
# ppi triples: Protein --PPI--> Protein  (STRING DB, combined_score >= 800)
ppi_raw_df = pd.read_csv('../../../raw/CGL/196627.protein.links.full.v12.0.txt', sep=' ')
ppi_df = ppi_raw_df[ppi_raw_df['combined_score'] >= 800]
for i in tqdm.tqdm(range(ppi_df.shape[0])):
    try:
        src_uni = id2uniid[ppi_df.iloc[i]['protein1'].split('.')[1]]
        dst_uni = id2uniid[ppi_df.iloc[i]['protein2'].split('.')[1]]
    except KeyError:
        continue
    triples.append(('CGL_Protein:' + src_uni, 'PPI', 'CGL_Protein:' + dst_uni))

100%|██████████| 19566/19566 [00:02<00:00, 7284.48it/s]


In [10]:
# protein GO triples
# NOTE: protein_go_annotation.txt has an extra 'GO NAME' column
prot_go_df = pd.read_csv('../../../raw/CGL/protein_go_annotation.txt', sep='\t')
prot_go_df = prot_go_df[prot_go_df['GENE PRODUCT ID'].isin(all_gene_uniid)]
prot_go_df = prot_go_df[prot_go_df['QUALIFIER'].isin(['enables', 'involved_in', 'part_of'])]
for i in range(prot_go_df.shape[0]):
    entity_id = prot_go_df.iloc[i]['GENE PRODUCT ID']
    qualifier  = prot_go_df.iloc[i]['QUALIFIER'].capitalize()
    go_term    = prot_go_df.iloc[i]['GO TERM']
    triples.append(('CGL_Protein:' + entity_id, qualifier, go_term))

In [11]:
# protein KO triples (KEGG)
# NOTE: CGL entries use Cgl-format locus tags (e.g. cgl:Cgl0001)
ko_df = pd.read_csv('../../../raw/shared/genes_ko.list', sep='\t', names=['gene', 'ko'])
ko_df = ko_df[ko_df['gene'].str.startswith('cgl:')]
for i in range(ko_df.shape[0]):
    try:
        prot_uni = cgl_id2uniid[ko_df.iloc[i]['gene'].split(':')[1]]
    except:
        continue
    ko_id = 'KO:' + ko_df.iloc[i]['ko'].split(':')[1]
    triples.append(('CGL_Protein:' + prot_uni, 'Has_KO_Item', ko_id))

In [12]:
# common_triples: Metabolite --Belongs_to--> Pathway, Pathway --in--> Pathway, etc.
# Filter: drop triples where any Metabolite: node is not in the original KG

# Collect all metabolite IDs present in existing KG triples
kg_mets = set()
for h, r, t in triples:
    if h.startswith('Metabolite:'):
        kg_mets.add(h)
    if t.startswith('Metabolite:'):
        kg_mets.add(t)

# Load and filter common_triples.txt
common_triples_df = pd.read_csv('../../../raw/shared/common_triples.txt', sep='\t', names=['from', 'rel', 'to'])

def _met_in_kg(row):
    for val in (row['from'], row['to']):
        if val.startswith('Metabolite:') and val not in kg_mets:
            return False
    return True

common_triples_filtered = common_triples_df[common_triples_df.apply(_met_in_kg, axis=1)]
print(f"common_triples: {len(common_triples_df)} total -> {len(common_triples_filtered)} after filtering out unknown mets")

for _, row in common_triples_filtered.iterrows():
    triples.append((row['from'], row['rel'], row['to']))

common_triples: 2919 total -> 1808 after filtering out unknown mets


### OUTPUT kg.txt

In [13]:
kg = pd.DataFrame(triples, columns=['from', 'rel', 'to'])
kg = kg.drop_duplicates()
print(kg.shape)
kg.head()

(30504, 3)


,from,rel,to
0,Reaction:ALAR,Produces,Metabolite:ala__D_c
1,Reaction:ALAR_-1,Consumes,Metabolite:ala__D_c
2,Reaction:ALAR,Consumes,Metabolite:ala__L_c
3,Reaction:ALAR_-1,Produces,Metabolite:ala__L_c
4,Reaction:ALATA_L,Produces,Metabolite:akg_c


In [14]:
kg.to_csv('../../../processed/kg/CGL/kg_reproduced.txt', sep='\t', index=False, header=False)